# DB Query Examples — Reference Notebook

This notebook demonstrates how to query the `matches` and `observations` tables using the project's database layer. Use it as a reference for common query patterns.

**Prerequisites:** Make sure your database is running (`docker-compose up -d`) and `.env.local` has a valid `POSTGRES_URL`.

## 1. Setup — Imports & Session

The `get_session()` context manager handles commit/rollback/close automatically.

In [ ]:
from db import get_session, Matches, Observations
from sqlalchemy import select, func, desc

## 2. Get a Match by Primary Key (`id`)

`session.get()` is the simplest way to fetch a single row by its primary key. Returns `None` if not found.

In [ ]:
MATCH_ID = 1  # Change this to a real ID from your database

with get_session() as session:
    match = session.get(Matches, MATCH_ID)
    
    if match:
        print(f"Found: {match}")
        print(f"  id:          {match.id}")
        print(f"  espn_id:     {match.espn_id}")
        print(f"  date:        {match.date}")
        print(f"  sport:       {match.sport}")
        print(f"  league:      {match.league}")
        print(f"  matchup:     {match.matchup}")
        print(f"  home_team:   {match.home_team}")
        print(f"  away_team:   {match.away_team}")
        print(f"  game_status: {match.game_status}")
        print(f"  poly_slug:   {match.poly_slug}")
    else:
        print(f"No match found with id={MATCH_ID}")

## 3. Get a Match by `espn_id`

`espn_id` is the unique external identifier from ESPN. Use `filter_by` for simple equality checks.

In [ ]:
ESPN_ID = "401656701"  # Change this to a real ESPN ID

with get_session() as session:
    match = session.query(Matches).filter_by(espn_id=ESPN_ID).first()
    
    if match:
        print(f"Found: {match}")
        print(f"  Matchup: {match.home_team} vs {match.away_team}")
        print(f"  Status:  {match.game_status}")
    else:
        print(f"No match found with espn_id={ESPN_ID}")

## 4. Get a Match by `poly_slug`

`poly_slug` is the unique Polymarket event slug, formatted as `{league}-{away_abbr}-{home_abbr}-{YYYY-MM-DD}`.

In [ ]:
POLY_SLUG = "mlb-stl-lad-2025-06-15"  # Change this to a real slug

with get_session() as session:
    match = session.query(Matches).filter_by(poly_slug=POLY_SLUG).first()
    
    if match:
        print(f"Found: {match}")
        print(f"  ESPN ID: {match.espn_id}")
        print(f"  Date:    {match.date}")
    else:
        print(f"No match found with poly_slug={POLY_SLUG}")

## 5. List All Matches

Fetch all rows from the `matches` table. Be careful with large tables.

In [ ]:
with get_session() as session:
    matches = session.query(Matches).all()
    
    print(f"Total matches: {len(matches)}\n")
    for m in matches:
        print(f"  [{m.id}] {m.matchup} | {m.game_status} | {m.date}")

## 6. Filter by `game_status`

Common statuses: `"Scheduled"`, `"In Progress"`, `"Final"`, `"Postponed"`, `"Cancelled"`.

In [ ]:
with get_session() as session:
    # Scheduled matches
    scheduled = session.query(Matches).filter_by(game_status="Scheduled").all()
    print(f"Scheduled matches: {len(scheduled)}")
    for m in scheduled:
        print(f"  [{m.id}] {m.matchup} | {m.date}")
    
    print()
    
    # Finished matches
    finished = session.query(Matches).filter_by(game_status="Final").all()
    print(f"Final matches: {len(finished)}")
    for m in finished:
        print(f"  [{m.id}] {m.matchup} | {m.date}")

## 7. Filter by `league` and `sport`

Chain multiple `filter_by` conditions or use `filter` for more complex expressions.

In [ ]:
with get_session() as session:
    # By league
    mlb_games = session.query(Matches).filter_by(league="mlb").all()
    print(f"MLB matches: {len(mlb_games)}")
    for m in mlb_games[:5]:  # Show first 5
        print(f"  [{m.id}] {m.matchup} | {m.game_status}")
    
    print()
    
    # By league + status
    nba_scheduled = session.query(Matches).filter_by(league="nba", game_status="Scheduled").all()
    print(f"Scheduled NBA matches: {len(nba_scheduled)}")
    for m in nba_scheduled[:5]:
        print(f"  [{m.id}] {m.matchup} | {m.date}")

## 8. Count Rows

Use `func.count()` for efficient counting without loading all rows.

In [ ]:
with get_session() as session:
    total = session.query(func.count(Matches.id)).scalar()
    scheduled_count = session.query(func.count(Matches.id)).filter_by(game_status="Scheduled").scalar()
    final_count = session.query(func.count(Matches.id)).filter_by(game_status="Final").scalar()
    
    print(f"Total matches:          {total}")
    print(f"Scheduled matches:      {scheduled_count}")
    print(f"Final matches:          {final_count}")

## 9. Order & Limit Results

Use `order_by` with `desc()` or `asc()` and `limit()` to control result size.

In [ ]:
with get_session() as session:
    # Most recent matches first
    recent = session.query(Matches).order_by(desc(Matches.date)).limit(10).all()
    print("10 most recent matches:")
    for m in recent:
        print(f"  [{m.id}] {m.matchup} | {m.date} | {m.game_status}")
    
    print()
    
    # Earliest matches first
    earliest = session.query(Matches).order_by(Matches.date).limit(5).all()
    print("5 earliest matches:")
    for m in earliest:
        print(f"  [{m.id}] {m.matchup} | {m.date} | {m.game_status}")

## 10. Access Related Observations

The `Matches` model has a relationship to `Observations`. Access them via `match.observations`.

In [ ]:
MATCH_ID = 1  # Change this to a real ID

with get_session() as session:
    match = session.get(Matches, MATCH_ID)
    
    if match:
        print(f"Match: {match.matchup}\n")
        print(f"Observations ({len(match.observations)}):")
        
        if not match.observations:
            print("  No observations yet.")
        
        for obs in match.observations:
            print(f"  [{obs.id}] {obs.observed_at} | source={obs.source}")
            print(f"    home_odds:      {obs.home_odds}")
            print(f"    away_odds:      {obs.away_odds}")
            print(f"    draw_odds:      {obs.draw_odds}")
            print(f"    over_under_line: {obs.over_under_line}")
            print(f"    over_odds:       {obs.over_odds}")
            print(f"    under_odds:      {obs.under_odds}")
            print()
    else:
        print(f"No match found with id={MATCH_ID}")

## 11. Query Observations Directly

You can also query the `observations` table directly and join back to `matches`.

In [ ]:
with get_session() as session:
    # All observations for a specific match
    observations = (
        session.query(Observations)
        .filter_by(match_id=1)  # Change to a real match_id
        .order_by(desc(Observations.observed_at))
        .all()
    )
    
    print(f"Observations for match_id=1: {len(observations)}")
    for obs in observations:
        print(f"  [{obs.id}] {obs.observed_at} | {obs.source} | home={obs.home_odds} away={obs.away_odds}")
    
    print()
    
    # Latest observation per match (most recent)
    latest = (
        session.query(Observations)
        .order_by(desc(Observations.observed_at))
        .limit(10)
        .all()
    )
    print("10 most recent observations:")
    for obs in latest:
        print(f"  [match_id={obs.match_id}] {obs.observed_at} | {obs.source} | home={obs.home_odds} away={obs.away_odds}")

## 12. Check if a Match Exists

Quick existence checks without loading the full row.

In [ ]:
ESPN_ID = "401656701"  # Change this

with get_session() as session:
    exists = session.query(Matches).filter_by(espn_id=ESPN_ID).first() is not None
    print(f"Match with espn_id={ESPN_ID} exists: {exists}")
    
    # Alternative using query().exists() — more efficient
    exists_sql = session.query(
        session.query(Matches).filter_by(espn_id=ESPN_ID).exists()
    ).scalar()
    print(f"(SQL EXISTS check): {exists_sql}")

## 13. Insert a Test Match (Upsert)

Use `MatchRepository.upsert_match()` to insert or update matches. On conflict (duplicate `espn_id`), it updates the existing row.

In [ ]:
from db.repository import MatchRepository
from datetime import datetime

test_match = [
    {
        "espn_id": "test-999",
        "date": datetime(2025, 6, 15, 19, 5),
        "sport": "baseball",
        "league": "mlb",
        "matchup": "Test Team A at Test Team B",
        "home_team": "Test Team B",
        "away_team": "Test Team A",
        "game_status": "Scheduled",
        "poly_slug": "mlb-tta-ttb-2025-06-15",
    }
]

with get_session() as session:
    repo = MatchRepository(session)
    repo.upsert_match(test_match)
    print("Upserted test match.")

# Verify it was inserted
with get_session() as session:
    match = session.query(Matches).filter_by(espn_id="test-999").first()
    if match:
        print(f"Verified: {match}")
        print(f"  ID: {match.id}")

## 14. Clean Up Test Data

Delete the test match we just inserted.

In [ ]:
from sqlalchemy import delete

with get_session() as session:
    result = session.execute(
        delete(Matches).where(Matches.espn_id == "test-999")
    )
    print(f"Deleted {result.rowcount} test match(es).")

---

## Quick Reference

| Task | Method |
|---|---|
| Get by PK | `session.get(Matches, id)` |
| Get by espn_id | `session.query(Matches).filter_by(espn_id="...").first()` |
| Get by poly_slug | `session.query(Matches).filter_by(poly_slug="...").first()` |
| Filter by status | `session.query(Matches).filter_by(game_status="Scheduled").all()` |
| Count rows | `session.query(func.count(Matches.id)).scalar()` |
| Order & limit | `session.query(Matches).order_by(desc(Matches.date)).limit(10).all()` |
| Check existence | `session.query(Matches).filter_by(espn_id="...").first() is not None` |
| Access observations | `match.observations` |
| Upsert matches | `MatchRepository(session).upsert_match([...])` |
| Delete | `session.execute(delete(Matches).where(...))` |